# 1) Finite Element Forward Problem

____
- **<u>Indicative duration</u>** : 1 hour 30 minutes

There are 4 types of code cells:
- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.
____

## A) First introductory example

- **<u>Indicative duration</u>** : 20 minutes

### i) Reference function

We consider the following analytical function supported on the unit square $\Omega$:
$$ u_0 : \left \{ \begin{array}{rcl}
\Omega = [0,1]^2 & \rightarrow & \mathbb R \\
(x , y) &\mapsto & (x-0.5)^2 + (y-0.5)^2
\end{array} \right.$$

We can define and vizualize such an analytical function in `NGSolve`.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
###############################################################################
## CODE CELL 1 : Define the geometry and reference function
###############################################################################

from ngsolve import unit_square, Mesh
mesh = Mesh(unit_square.GenerateMesh(maxh = 0.1)) # create the discretized geometry

from ngsolve import x, y
u0 = (x-0.5)**2 + (y-0.5)**2                      # define the analytic function

from ngsolve.webgui import Draw
Draw(u0, mesh,                                    # Draw the graph of the analytical function
     settings = { "Objects" : { "Wireframe" : False }, "deformation" :  1})

_____
### ii) Discretization

In general, we don't have access to the analytical expression of the function. We therefore use approximations that are computable with a finite number of parameters called **degrees of freedom** (dofs). 

The first step is to discretize the geometry into a finite set of cells, lines and vertices called a **mesh**.

- In 2D, we focus on **triangular meshes**, but there exist a wide variety of meshes (structured, unstructured, quadrangular, etc.)

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
###############################################################################
## CODE CELL 2 : Display the mesh
###############################################################################

Draw(mesh)    # draw the mesh defined in CODE CELL 1

____
### iii) Function spaces

The second step to define discretized function approximation is to chose an appropriate function space.
We focus on two types of function space:

- $ L^2(\Omega) = \left \{ v : \Omega \rightarrow \mathbb R, \int_\Omega |v|^2 \leq \infty \right \}$, discretized by **<u>discontinuous</u>** functions defined piecewise on **each cell** (element) of the mesh.

-  $ H^1(\Omega) = \left \{ v \in L^2(\Omega), \nabla v \in L^2(\Omega) \right \}$, discretized by **<u>continuous</u>** functions defined on **each node** (vertex) of the mesh.

A discretized function is then reduced to a **polynomial** into each cell of the mesh. The **degree** of this polynomial can be specified and is set up by the parameter `order`.

Other types of spaces can be defined, especially for vector fields having only *tangential* or *normal* continuity accross interfaces; see [this page](https://docu.ngsolve.org/v6.2.2103/i-tutorials/unit-2.3-hcurlhdiv/hcurlhdiv.html) for more information.

|**<font color='orange'>[RUN & PLAY]</font>**|
|---|

In [ ]:
###############################################################################
## CODE CELL 3 : Define and display GridFunctions 
###############################################################################

from ngsolve import H1, L2, GridFunction

polynomial_degree = 1
fes = H1(mesh, order=polynomial_degree)     # define the discretized function space on the mesh
#fes = L2(mesh, order=polynomial_degree)

gfu = GridFunction(fes)                     # a GridFunction is a discretized function on a given function space
gfu.vec.data[-1] = 1
Draw(gfu, mesh,                             # Draw the graph of the grid function
     settings = { "deformation" :  0.5})

_____
### iv) Least square formulation

A way to approximate any function on a discretized finite element space is to find $u\in H$ (with $H=L^2(\Omega)$ or $H^1(\Omega)$) **minimizing the integral of the square error** with the reference $u_0$: 

$$ u = \arg \min_{v\in H} J(v) = \frac{1}{2}\int_{x\in\Omega} ( v(x) -u_0 )^2 \; \mathrm{d} x$$


In what follows, we drop the $x$ dependency for conciseness :
- $u(x)\rightarrow u$
- no "$ x\in $"  and "$\mathrm{d}x$" in the integrals that are always evaluated on the geometric space $\Omega$.
  
|**<font color='red'>[FILL & RUN]</font>**|
|---|

In [ ]:
from ngsolve import Integrate

def J(u):
    """ integral of the half squared error """
    return Integrate( u, u.space.mesh)